# cerberus-neuro — sanity-check diagnostics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/02_sanity_check.ipynb)

Run before re-attempting the full paired multi-task training in `02_train.ipynb`. Three diagnostics:

1. **Data pipeline visualization.** Render 5 random training samples with all 6 channels labeled to confirm channel order, augmentation, and labels look right.
2. **Cell-type-only training (5 epochs).** Strip the multi-task model down to just the encoder + cell-type head. If `acc_cell_type > 0.8` after 5 epochs, the encoder + data pipeline are sound and the multi-task interference is the problem in `02_train.ipynb`. If not, we have a data or pipeline bug.
3. **Virtual-staining-only training (5 epochs).** Encoder + U-Net decoder, no classifier heads. Tells us if the decoder architecture can learn at all on this data. After training, render predicted vs target fluorescence to verify the decoder isn't collapsed to constant output.

Total runtime on Colab Pro L4: ~30–45 min, mostly the two short training runs. Data is reused from `/content/cerberus-cache/` populated by `02_train.ipynb`.

## 1. Setup

In [ ]:
!pip install -q --upgrade "cerberus-neuro[training] @ git+https://github.com/PatrickJReed/cerberus-neuro.git@main" imagecodecs

import sys
for _m in list(sys.modules):
    if _m == 'cerberus_neuro' or _m.startswith('cerberus_neuro.'):
        del sys.modules[_m]
print('install + cache purge done')

In [ ]:
import os
from pathlib import Path
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_BASE = Path('/content/drive/MyDrive/cerberus-neuro/cache/v0')
    CACHE_DIR = Path('/content/cerberus-cache')
except Exception:
    CKPT_BASE = Path.home() / '.cache' / 'cerberus-neuro' / 'v0'
    CACHE_DIR = CKPT_BASE

CKPT_BASE.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'CACHE_DIR (fast):  {CACHE_DIR}')
print(f'CKPT_BASE (drive): {CKPT_BASE}')
print(f'GPU: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from cerberus_neuro.data import (
    build_manifest, subset_manifest, well_level_split, NeuroPaintingDataset,
    CHANNEL_ORDER,
)

manifest = build_manifest(CACHE_DIR)
subset = subset_manifest(manifest, wells_per_cell_type=48, sites_per_well=9, seed=0)
train_manifest, val_manifest = well_level_split(subset, val_frac=0.2, seed=0)
print(f'train: {len(train_manifest)} sites, val: {len(val_manifest)} sites')

## 2. Data pipeline visualization

Render 5 random training samples with all 6 channels and labels. Check:

- Channel order matches `CHANNEL_ORDER` (BF, DNA, Mito, AGP, ER, RNA).
- DNA channel shows nuclei (bright punctate spots).
- BF shows whole cells with visible boundaries.
- Cell-type label matches what the morphology suggests.

In [ ]:
import itertools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Pick one site from each (cell_type, condition) cell so the viz covers all 8
# strata. Without this, crops_per_site=4 plus shuffle would yield 4 crops from
# the same site (same labels, same morphology) and a 5th from the next site,
# which is uninformative.
strata_rows = []
for ct in ['stem', 'progen', 'neuron', 'astro']:
    for cond in ['control', 'deletion']:
        matching = train_manifest[
            (train_manifest['Metadata_cell_type'] == ct)
            & (train_manifest['Metadata_line_condition'] == cond)
        ]
        if len(matching):
            strata_rows.append(matching.sample(1, random_state=42).iloc[0])
mini_manifest = pd.DataFrame(strata_rows).reset_index(drop=True)
print(f'sampling 1 site per (cell_type, condition); {len(mini_manifest)} sites total:')
print(mini_manifest[['Metadata_cell_type', 'Metadata_line_condition', 'Metadata_Plate', 'Metadata_Well', 'Metadata_Site']].to_string(index=False))

viz_ds = NeuroPaintingDataset(
    mini_manifest, CACHE_DIR,
    crop_size=256, crops_per_site=1,
    min_cells_per_crop=1, augment=False, shuffle=False,
)
samples = list(viz_ds)

CELL_TYPES_DISPLAY = ['stem', 'progen', 'neuron', 'astro']
COND_DISPLAY = ['control', 'deletion']
CH_NAMES = ['BF', 'DNA', 'Mito', 'AGP', 'ER', 'RNA']

fig, axes = plt.subplots(len(samples), 6, figsize=(18, 3 * len(samples)))
if len(samples) == 1:
    axes = axes[None, :]
for row, (bf, fluo, ct, cond) in enumerate(samples):
    label = f'{CELL_TYPES_DISPLAY[ct]} / {COND_DISPLAY[cond]}'
    all_ch = torch.cat([bf, fluo], dim=0).numpy()
    for col, name in enumerate(CH_NAMES):
        ax = axes[row, col]
        img = all_ch[col]
        lo, hi = np.percentile(img, [1, 99])
        ax.imshow(np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1), cmap='gray')
        if row == 0:
            ax.set_title(name, fontsize=10, fontweight='bold')
        if col == 0:
            ax.set_ylabel(label, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
plt.tight_layout()
plt.show()


## 3. Cell-type-only training (5 epochs)

Single-head model, just the encoder + a 4-class softmax. Same data, same augmentation, same hyperparameters as the multi-task setup. If cell-type accuracy doesn't reach 0.8+ in 5 epochs, the encoder + data are not the source of the multi-task model's regression. If it does, the multi-task setup is the issue.

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

from cerberus_neuro.model import CellTypeOnlyModel

torch.set_float32_matmul_precision('high')
torch.manual_seed(0)

BATCH_SIZE = 64
NUM_WORKERS = 8
CROPS_PER_SITE = 12

train_ds = NeuroPaintingDataset(
    train_manifest, CACHE_DIR,
    crop_size=256, crops_per_site=CROPS_PER_SITE,
    min_cells_per_crop=1, augment=True,
)
val_ds = NeuroPaintingDataset(
    val_manifest, CACHE_DIR,
    crop_size=256, crops_per_site=CROPS_PER_SITE,
    min_cells_per_crop=1, augment=False, shuffle=False,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          persistent_workers=True, pin_memory=True, prefetch_factor=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          persistent_workers=True, pin_memory=True, prefetch_factor=4)

model = CellTypeOnlyModel().to(device)
encoder_params = list(model.encoder.parameters())
head_params = list(model.head.parameters())
optimizer = AdamW([
    {'params': encoder_params, 'lr': 3e-5},
    {'params': head_params, 'lr': 3e-4},
], weight_decay=1e-4)

N_EPOCHS = 5
steps_per_epoch = max(1, (len(train_manifest) * CROPS_PER_SITE) // BATCH_SIZE)
total_steps = N_EPOCHS * steps_per_epoch
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

print(f'steps_per_epoch={steps_per_epoch}  total_steps={total_steps}')

step = 0
for epoch in range(N_EPOCHS):
    model.train()
    train_loss = train_correct = train_n = 0
    for bf, fluo, ct, cond in train_loader:
        bf, ct = bf.to(device, non_blocking=True), ct.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        ctx = torch.amp.autocast('cuda') if scaler is not None else torch.amp.autocast('cuda', enabled=False)
        with ctx:
            logits = model(bf)
            loss = F.cross_entropy(logits, ct)
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        train_loss += loss.item() * bf.size(0)
        train_correct += (logits.argmax(1) == ct).sum().item()
        train_n += bf.size(0)
        step += 1
        if step % 25 == 0:
            print(f'  step {step:4d}: loss={loss.item():.4f}  acc={(logits.argmax(1) == ct).float().mean().item():.4f}')
        if step >= (epoch + 1) * steps_per_epoch:
            break

    model.eval()
    val_correct = val_n = 0
    val_loss_sum = 0.0
    with torch.no_grad():
        for bf, fluo, ct, cond in val_loader:
            bf, ct = bf.to(device), ct.to(device)
            logits = model(bf)
            val_loss_sum += F.cross_entropy(logits, ct).item() * bf.size(0)
            val_correct += (logits.argmax(1) == ct).sum().item()
            val_n += bf.size(0)
    print(f'epoch {epoch}: train_loss={train_loss/train_n:.4f}  '
          f'train_acc={train_correct/train_n:.4f}  '
          f'val_loss={val_loss_sum/val_n:.4f}  val_acc={val_correct/val_n:.4f}')

## 4. Virtual-staining-only training (5 epochs)

Encoder + U-Net decoder, no classifier heads. Loss is `0.85 * L1 + 0.15 * (1 - SSIM)` averaged across the 5 fluorescence channels. After 5 epochs we'll render predicted vs target fluorescence for a few val samples; if predictions are visibly cell-shaped (not constant gray), the decoder is learning.

In [ ]:
# Self-contained: this cell can run after a kernel restart provided § 1 setup
# (manifests, CACHE_DIR, device) has run.
import gc
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

from cerberus_neuro.model import VirtualStainingOnlyModel
from cerberus_neuro.training import virtual_staining_loss
from pytorch_msssim import ssim

torch.set_float32_matmul_precision('high')
torch.manual_seed(0)

# Free any leftover state from § 3 (cell-type-only) if it ran in this kernel.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# VS-only config (independent of § 3 settings).
VS_BATCH = 32                # smaller than § 3's 64 since UNet decoder is heavier
VS_NUM_WORKERS = 4
VS_CROPS_PER_SITE = 12
VS_N_EPOCHS = 5

vs_train_ds = NeuroPaintingDataset(
    train_manifest, CACHE_DIR,
    crop_size=256, crops_per_site=VS_CROPS_PER_SITE,
    min_cells_per_crop=1, augment=True,
)
vs_val_ds = NeuroPaintingDataset(
    val_manifest, CACHE_DIR,
    crop_size=256, crops_per_site=VS_CROPS_PER_SITE,
    min_cells_per_crop=1, augment=False, shuffle=False,
)
vs_train_loader = DataLoader(
    vs_train_ds, batch_size=VS_BATCH,
    num_workers=VS_NUM_WORKERS, persistent_workers=False,
    pin_memory=True, prefetch_factor=2,
)
vs_val_loader = DataLoader(
    vs_val_ds, batch_size=VS_BATCH,
    num_workers=VS_NUM_WORKERS, persistent_workers=False,
    pin_memory=True, prefetch_factor=2,
)

vs_model = VirtualStainingOnlyModel().to(device)
encoder_params = list(vs_model.encoder.parameters())
decoder_params = list(vs_model.decoder.parameters())
vs_opt = AdamW([
    {'params': encoder_params, 'lr': 3e-5},
    {'params': decoder_params, 'lr': 3e-4},
], weight_decay=1e-4)

vs_steps_per_epoch = max(1, (len(train_manifest) * VS_CROPS_PER_SITE) // VS_BATCH)
vs_total_steps = VS_N_EPOCHS * vs_steps_per_epoch
vs_sched = CosineAnnealingLR(vs_opt, T_max=vs_total_steps)
vs_scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

print(f'VS-only steps_per_epoch={vs_steps_per_epoch}  total_steps={vs_total_steps}')

step = 0
FLUO_NAMES = ['DNA', 'Mito', 'AGP', 'ER', 'RNA']
for epoch in range(VS_N_EPOCHS):
    vs_model.train()
    train_loss_sum = train_n = 0
    for bf, fluo, ct, cond in vs_train_loader:
        bf, fluo = bf.to(device, non_blocking=True), fluo.to(device, non_blocking=True)
        vs_opt.zero_grad(set_to_none=True)
        ctx = torch.amp.autocast('cuda') if vs_scaler is not None else torch.amp.autocast('cuda', enabled=False)
        with ctx:
            pred = vs_model(bf)
            loss = virtual_staining_loss(pred, fluo)
        if vs_scaler is not None:
            vs_scaler.scale(loss).backward()
            vs_scaler.unscale_(vs_opt)
            torch.nn.utils.clip_grad_norm_(vs_model.parameters(), 1.0)
            vs_scaler.step(vs_opt)
            vs_scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(vs_model.parameters(), 1.0)
            vs_opt.step()
        vs_sched.step()

        train_loss_sum += loss.item() * bf.size(0)
        train_n += bf.size(0)
        step += 1
        if step % 25 == 0:
            print(f'  step {step:4d}: loss={loss.item():.4f}')
        if step >= (epoch + 1) * vs_steps_per_epoch:
            break

    vs_model.eval()
    val_loss_sum = val_n = 0
    per_ch_l1 = torch.zeros(5)
    per_ch_ssim = torch.zeros(5)
    with torch.no_grad():
        for bf, fluo, ct, cond in vs_val_loader:
            bf, fluo = bf.to(device), fluo.to(device)
            pred = vs_model(bf)
            val_loss_sum += virtual_staining_loss(pred, fluo).item() * bf.size(0)
            per_ch_l1 += (pred - fluo).abs().mean(dim=(0, 2, 3)).cpu() * bf.size(0)
            for c in range(5):
                s = ssim(pred[:, c:c+1].float(), fluo[:, c:c+1].float(),
                         data_range=1.0, size_average=True)
                per_ch_ssim[c] += s.item() * bf.size(0)
            val_n += bf.size(0)
    print(f'epoch {epoch}: train_loss={train_loss_sum/train_n:.4f}  val_loss={val_loss_sum/val_n:.4f}')
    for c, name in enumerate(FLUO_NAMES):
        print(f'    {name}: L1={per_ch_l1[c].item()/val_n:.4f}  SSIM={per_ch_ssim[c].item()/val_n:.4f}')


## 5. Visualize VS-only predictions

Render 3 random val samples with brightfield input, predicted fluorescence per channel, and ground-truth fluorescence per channel. If predicted images are uniform gray (constant ~0.5), the decoder is collapsed and not learning. If they look like blurry blobs that vaguely track the BF cells, the decoder is learning slowly. If they show recognizable nuclei in DNA / structure in Mito etc., the decoder is working.

In [ ]:
vs_model.eval()
viz_loader = DataLoader(vs_val_ds, batch_size=4, num_workers=0)
bf_v, fluo_v, ct_v, cond_v = next(iter(viz_loader))
bf_v, fluo_v = bf_v.to(device), fluo_v.to(device)
with torch.no_grad():
    pred_v = vs_model(bf_v)

n_show = 3
fig, axes = plt.subplots(n_show, 11, figsize=(22, 6.5))
for row in range(n_show):
    bf_img = bf_v[row, 0].cpu().numpy()
    lo, hi = np.percentile(bf_img, [1, 99])
    axes[row, 0].imshow(np.clip((bf_img - lo) / max(hi - lo, 1e-6), 0, 1), cmap='gray')
    axes[row, 0].set_title('BF', fontsize=9, fontweight='bold' if row == 0 else 'normal')
    for c, name in enumerate(FLUO_NAMES):
        pred_img = pred_v[row, c].cpu().numpy()
        target_img = fluo_v[row, c].cpu().numpy()
        # Use target's intensity scaling for both so they're directly comparable.
        lo, hi = np.percentile(target_img, [1, 99])
        norm = lambda im: np.clip((im - lo) / max(hi - lo, 1e-6), 0, 1)
        axes[row, 1 + 2*c].imshow(norm(pred_img), cmap='gray')
        axes[row, 2 + 2*c].imshow(norm(target_img), cmap='gray')
        if row == 0:
            axes[row, 1 + 2*c].set_title(f'pred {name}', fontsize=9)
            axes[row, 2 + 2*c].set_title(f'true {name}', fontsize=9)
    for col in range(11):
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
plt.tight_layout()
plt.show()

## Interpretation guide

**Cell-type-only training:**
- val_acc → 0.80+ by epoch 4: encoder + data are fine. Multi-task interference is the v0 culprit. Move to discriminative-LR fix in `02_train.ipynb`.
- val_acc stuck at <0.7: data pipeline or label issue. Investigate the visualization in § 2 and the data subsetting logic in `data.py`.

**Virtual-staining-only training:**
- DNA SSIM > 0.4 by epoch 4 with visibly nucleus-shaped predicted images: U-Net decoder works fine on its own. Multi-task is the issue.
- DNA SSIM stuck at <0.2 with constant-gray predictions: decoder is collapsed. Likely architectural problem — investigate sigmoid saturation, skip-connection magnitude, or replace U-Net decoder with a simpler upsample-conv stack.

**Both single-task setups work in isolation but multi-task doesn't:**
- The Cerberus shared-encoder framing is incompatible with this data scale at 20× brightfield. v0 should pivot to two separate models (one for cell type, one for virtual staining) with the same encoder architecture but no shared training. Per-cell-crop pipeline (v1) becomes the path forward for unified multi-task.